# Employee Data — Bronze Layer Ingestion
### Raw CSV to Delta Lake (Medallion Architecture)

**Purpose:** This notebook ingests raw CSV files from the ADLS source container into the **bronze layer** of the medallion architecture. Each source file is read as-is, enriched with audit metadata, and persisted as a Delta table for downstream cleansing in the silver layer.

**What happens here:**
1. Read all 13 CSV source files from `abfss://employee@.../source/`
2. Append audit columns to every table: `_bronze_ingested_at`, `_bronze_source_file`, `_bronze_batch_id`, `_bronze_is_valid`
3. Write each DataFrame as a Delta table under `bronze/` and register it in the `bronze_employee` schema

**Source Files (13 tables):**

| Category | Tables |
|----------|--------|
| Core Entities | Employee, District, Admin, Teacher, Other_Employee |
| Compensation | Total_PAB, PAB_Lineitem, PAB_item, REG_pay, OT_Pay, Benefit |
| Certifications | Certification, Teacher_Cert |

**Audit Columns Added:**
- `_bronze_ingested_at` — timestamp of ingestion
- `_bronze_source_file` — full ADLS path of the source CSV
- `_bronze_batch_id` — unique UUID per row (batch traceability)
- `_bronze_is_valid` — basic non-null check on the first column

---
_Downstream: [employee_silver](#) cleanses and conforms these tables into dimension/fact models._

In [0]:
# ── PySpark & Delta imports ──
# functions → column-level operations (timestamps, nulls, UUIDs)
# DeltaTable → kept in scope for potential merge/upsert patterns later

from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
# ── Medallion architecture paths (ADLS Gen2) ──
# Root container: "employee" on dataanlysisazuredatalake
# Each layer (bronze / silver / gold) gets its own sub-folder and UC schema.

root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/"
bronze_path = f"{root_path}/bronze"
silver_path = f"{root_path}/silver"
gold_path = f"{root_path}/gold"

# ── Unity Catalog schema names ──
bronze_sch = "bronze_employee"
silver_sch = "silver_employee"
gold_sch = "gold_employee"
employee_db = "employeedatacatalog"

# ── Create all three schemas if they don't already exist ──
# This ensures the catalog structure is in place before any writes happen.
schema_data = [bronze_sch, silver_sch, gold_sch]

for sch in schema_data:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {employee_db}.{sch}")


In [0]:
# ============================================================
# STEP 1: Discover and read all CSV files from the source folder
# ============================================================
# The source container holds 13 CSV files — one per entity.
# We dynamically list them, infer schemas, and store each as
# a DataFrame in a dictionary keyed by the cleaned file name.

source_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net/source/"

# List all files and filter to only .csv extensions
csv_files = [file for file in dbutils.fs.ls(source_path) if file.name.endswith('.csv')]

# Read each CSV into a dictionary of DataFrames
# Key = file name without extension (spaces replaced with underscores)
dataframes = {}

for file in csv_files:
    name = file.name.replace('.csv', '').replace(' ', '_')
    dataframes[name] = spark.read.option("header", True).option("inferSchema", True).csv(file.path)

print(f"Loaded {len(dataframes)} tables: {list(dataframes.keys())}")

In [0]:
# ============================================================
# STEP 2: Add bronze-layer audit/metadata columns to every table
# ============================================================
# These four columns enable downstream traceability and data quality:
#   _bronze_ingested_at  → when the row was ingested (processing timestamp)
#   _bronze_source_file  → full ADLS path of the original CSV (from Spark metadata)
#   _bronze_batch_id     → a UUID per row to uniquely tag this ingestion batch
#   _bronze_is_valid     → simple non-null check on the first column (quick quality flag)

bronze_dfs = {}

for name, df in dataframes.items():
    first_col = df.columns[0]   # Use the first column as the validity check target
    
    bronze_dfs[name] = df.withColumn("_bronze_ingested_at", f.current_timestamp())\
                         .withColumn("_bronze_source_file", f.col("_metadata.file_path"))\
                         .withColumn("_bronze_batch_id", f.expr("uuid()"))\
                         .withColumn("_bronze_is_valid", f.col(first_col).isNotNull())

# Print a summary of what was enriched
for name, df in bronze_dfs.items():
    print(f"{name}: {len(df.columns)} columns (validity check on '{df.columns[0]}')")


In [0]:
# ============================================================
# STEP 3: Write each DataFrame as a Delta table and register in UC
# ============================================================
# For each table we:
#   1. Write to the bronze ADLS path as Delta format (overwrite mode)
#   2. Register it as an external table in the bronze_employee schema
# This makes the raw data queryable via SQL and ready for silver-layer cleansing.

for name, df in bronze_dfs.items():
    table_path = f"{bronze_path}/{name.lower()}"
    
    # Write the DataFrame as a Delta table
    df.write.format("delta").mode("overwrite").save(table_path)

    # Register in Unity Catalog so it's accessible via SQL
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {employee_db}.{bronze_sch}.{name}
                  USING DELTA LOCATION '{table_path}'""")
    print(f"Saved: {employee_db}.{bronze_sch}.{name}")